In [3]:
# import os
# import json
# import gc
# import math
# import multiprocess as mp
# from datasets import load_dataset
# from transformers import AutoTokenizer

# def worker_task(worker_id, gpu_id, chunk_data, model_path, output_dir):
#     import os
#     import json
#     import gc
#     import torch
#     from vllm import LLM, SamplingParams
#     from transformers import AutoTokenizer
#     import time
    
#     os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu_id)
    
#     # 2. CHỈ IMPORT vLLM VÀ TORCH Ở TRONG NÀY ĐỂ TRÁNH LỖI CUDA CONTEXT
#     import torch
#     from vllm import LLM, SamplingParams

#     chunk_file = os.path.join(output_dir, f"chunk_{worker_id}.jsonl")
#     print(f"[Worker {worker_id} | GPU {gpu_id}] Bắt đầu xử lý {len(chunk_data)} câu...")

#     # ==========================================
#     # TIỀN XỬ LÝ TOKEN (Giảm tải cho CPU lúc inference)
#     # ==========================================
#     tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
#     prompts_all_ids = []
    
#     for prompt in chunk_data:
#         conversation = [
#             {"role": "system", "content": "You are a teacher. Solve the problem and put your final answer within \\boxed{}."},
#             {"role": "user", "content": prompt}
#         ]
#         formatted_prompt_ids = tokenizer.apply_chat_template(
#             conversation,
#             add_generation_prompt=True,
#             tokenize=True # Trực tiếp lấy Token IDs
#         )
#         prompts_all_ids.append(formatted_prompt_ids)

#     delay = worker_id * 15
#     print(f"⏳ [Worker {worker_id}] Đang chờ {delay} giây để tránh kẹt VRAM...")
#     time.sleep(delay)

#     # ==========================================
#     # KHỞI TẠO vLLM
#     # ==========================================
#     llm = LLM(
#         model=model_path,
#         trust_remote_code=True,
#         dtype="bfloat16",
#         tensor_parallel_size=1, # Mỗi instance chạy trên 1 card
#         gpu_memory_utilization=0.43, # QUAN TRỌNG: Chia đôi VRAM cho 2 instance
#         seed=42,
#         max_model_len=2048
#     )

#     sampling_params = SamplingParams(
#         temperature=0.85,
#         top_p=0.95,
#         max_tokens=512,
#         skip_special_tokens=True
#     )

#     # ==========================================
#     # CHẠY INFERENCE
#     # ==========================================
#     outputs = llm.generate(prompt_token_ids=prompts_all_ids, sampling_params=sampling_params)

#     # LƯU KẾT QUẢ CỦA CHUNK NÀY
#     output_data = []
#     for i, output in enumerate(outputs):
#         generated_text = output.outputs[0].text.strip()
#         output_data.append({
#             'prompt': chunk_data[i],
#             'generated_text': generated_text,
#         })

#     with open(chunk_file, 'w', encoding='utf-8') as f:
#         for item in output_data:
#             f.write(json.dumps(item, ensure_ascii=False) + "\n")

#     print(f"✅ [Worker {worker_id} | GPU {gpu_id}] Hoàn thành! Đã lưu {chunk_file}")
    
#     del llm
#     torch.cuda.empty_cache()
#     gc.collect()


# def run_parallel_vllm(model_path, output_dir, output_file, num_gpus, instances_per_gpu=2):
#     """
#     Hàm chính để quản lý tiến trình.
#     """
#     os.makedirs(output_dir, exist_ok=True)
    
#     # Bắt buộc cho multiprocessing với PyTorch/CUDA
#     try:
#         mp.set_start_method('spawn', force=True)
#     except RuntimeError:
#         pass

#     # 1. Tải dữ liệu
#     print("⏳ Loading dataset in Main Process...")
#     data = load_dataset('VoCuc/MetaMathQA-50k-256', split='train')['query']
    
#     # 2. Tính toán chia chunk
#     total_workers = num_gpus * instances_per_gpu
#     chunk_size = math.ceil(len(data) / total_workers)
#     chunks = [data[i:i + chunk_size] for i in range(0, len(data), chunk_size)]

#     print(f"\n{'='*60}")
#     print(f"🚀 KHỞI ĐỘNG HỆ THỐNG PHÂN TÁN")
#     print(f"📊 Tổng số GPU: {num_gpus}")
#     print(f"👯 Số instance trên mỗi GPU: {instances_per_gpu}")
#     print(f"🔥 Tổng số tiến trình (Workers): {total_workers}")
#     print(f"📦 Mỗi tiến trình xử lý khoảng: {chunk_size} câu")
#     print(f"{'='*60}\n")

#     # 3. Khởi tạo và chạy các tiến trình
#     processes = []
#     for worker_id in range(total_workers):
#         gpu_id = worker_id % num_gpus # Xoay vòng GPU (VD: 2 GPU thì ID là 0, 1, 0, 1)
#         chunk_data = chunks[worker_id]
        
#         p = mp.Process(
#             target=worker_task, 
#             args=(worker_id, gpu_id, chunk_data, model_path, output_dir)
#         )
#         processes.append(p)
#         p.start()

#     # 4. Chờ tất cả các worker chạy xong
#     for p in processes:
#         p.join()

#     # 5. Gộp các file chunk lại thành 1 file JSONL duy nhất
#     print("\n🔄 Đang gộp kết quả từ các workers...")
#     final_output_path = os.path.join(output_dir, output_file)
    
#     with open(final_output_path, 'w', encoding='utf-8') as outfile:
#         for worker_id in range(total_workers):
#             chunk_file = os.path.join(output_dir, f"chunk_{worker_id}.jsonl")
#             if os.path.exists(chunk_file):
#                 with open(chunk_file, 'r', encoding='utf-8') as infile:
#                     outfile.write(infile.read())
#                 os.remove(chunk_file) # Xóa file rác sau khi gộp xong

#     print(f"🎉 HOÀN THÀNH TOÀN BỘ! File cuối cùng: {final_output_path}")


In [4]:
# models_to_test = [
#     # {
#     #     "model_path": "Qwen/Qwen2.5-Math-1.5B-Instruct",
#     #     "output_dir": "data/dpo/Qwen/Qwen2.5-Math-1.5B-Instruct",
#     #     "output_file": "generated_train.jsonl"
#     # },
#     {
#         "model_path": "Qwen/Qwen2.5-0.5B",
#         "output_dir": "data/dpo/Qwen/Qwen2.5-0.5B",
#         "output_file": "generated_train.jsonl"
#     }
# ]

# for config in models_to_test:
#     run_parallel_vllm(
#         model_path=config["model_path"],
#         output_dir=config["output_dir"],
#         output_file=config["output_file"],
#         num_gpus=2
#     )

In [5]:
import os
import json
import gc
import torch
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from datasets import load_dataset

# os.environ["CUDA_VISIBLE_DEVICES"] = "0"

def run_vllm_inference(model_path, output_dir, output_file, tensor_parallel_size=1):
    """
    Hàm chạy inference tự động cho bất kỳ model nào.
    """
    print(f"\n{'='*60}")
    print(f"🚀 BẮT ĐẦU CHẠY MODEL: {model_path}")
    print(f"📂 Thư mục lưu: {output_dir}/{output_file}")
    print(f"{'='*60}")
    
    os.makedirs(output_dir, exist_ok=True)

    # ==========================================
    # 1. CHUẨN BỊ DỮ LIỆU & PROMPT
    # ==========================================
    print("⏳ Loading tokenizer and dataset...")
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    data = load_dataset('VoCuc/MetaMathQA-50k-256', split='train')['query']

    print("🧩 Applying chat template...")
    prompts_all = []
    for prompt in data:
        conversation = [
            {"role": "system", "content": "You are a teacher. Solve the problem and put your final answer within \\boxed{}."},
            {"role": "user", "content": prompt}
        ]
        formatted_prompt = tokenizer.apply_chat_template(
            conversation,
            add_generation_prompt=True,
            tokenize=False
        )
        prompts_all.append(formatted_prompt)

    # ==========================================
    # 2. KHỞI TẠO vLLM ENGINE
    # ==========================================
    print("🔥 Initializing vLLM...")
    llm = LLM(
        model=model_path,
        trust_remote_code=True,
        dtype="bfloat16",
        tensor_parallel_size=tensor_parallel_size,
        gpu_memory_utilization=0.9,
        seed=42
    )

    # ==========================================
    # 3. CHẠY INFERENCE
    # ==========================================
    sampling_params = SamplingParams(
        temperature=0.85,
        top_p=0.95,
        max_tokens=512,
        skip_special_tokens=True
    )

    print(f"⚙️ Generating responses for {len(prompts_all)} prompts...")
    outputs = llm.generate(prompts_all, sampling_params)

    # ==========================================
    # 4. LƯU KẾT QUẢ
    # ==========================================
    output_data = []
    for i, output in enumerate(outputs):
        generated_text = output.outputs[0].text.strip()
        output_data.append({
            'prompt': data[i],
            'generated_text': generated_text,
        })

    output_path = os.path.join(output_dir, output_file)
    with open(output_path, 'w', encoding='utf-8') as f:
        for item in output_data:
            f.write(json.dumps(item, ensure_ascii=False) + "\n")

    print(f"✅ Đã lưu xong file tại: {output_path}")

    print("🧹 Đang dọn dẹp bộ nhớ GPU...")
    # Phá hủy đối tượng LLM để PyTorch biết vùng nhớ này không còn dùng nữa
    del llm
    del tokenizer
    # Gọi Garbage Collector của Python
    gc.collect()
    # Ép CUDA dọn dẹp các cache đang chiếm giữ
    torch.cuda.empty_cache()
    print("✨ Sẵn sàng cho model tiếp theo!\n")



In [6]:
models_to_test = [
    # {
    #     "model_path": "Qwen/Qwen2.5-Math-1.5B-Instruct",
    #     "output_dir": "data/dpo/Qwen/Qwen2.5-Math-1.5B-Instruct",
    #     "output_file": "generated_train.jsonl"
    # },
    # {
    #     "model_path": "Qwen/Qwen2.5-0.5B",
    #     "output_dir": "data/dpo/Qwen/Qwen2.5-0.5B",
    #     "output_file": "generated_train.jsonl"
    # },
    {
        "model_path": "Qwen/Qwen2.5-Math-7B-Instruct",
        "output_dir": "data/dpo/Qwen/Qwen2.5-Math-7B-Instruct",
        "output_file": "generated_train.jsonl"
    }
    
]

for config in models_to_test:
    run_vllm_inference(
        model_path=config["model_path"],
        output_dir=config["output_dir"],
        output_file=config["output_file"],
        tensor_parallel_size=2
    )


🚀 BẮT ĐẦU CHẠY MODEL: Qwen/Qwen2.5-Math-7B-Instruct
📂 Thư mục lưu: data/dpo/Qwen/Qwen2.5-Math-7B-Instruct/generated_train.jsonl
⏳ Loading tokenizer and dataset...


🧩 Applying chat template...
🔥 Initializing vLLM...
INFO 05-09 09:34:03 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'seed': 42, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-Math-7B-Instruct'}
INFO 05-09 09:34:15 [nixl_utils.py:20] Setting UCX_RCACHE_MAX_UNRELEASED to '1024' to avoid a rare memory leak in UCX when using NIXL.
WARNING 05-09 09:34:15 [nixl_utils.py:34] NIXL is not available
WARNING 05-09 09:34:15 [nixl_utils.py:44] NIXL agent config is not available
INFO 05-09 09:34:15 [model.py:555] Resolved architecture: Qwen2ForCausalLM
INFO 05-09 09:34:15 [model.py:1680] Using max model len 4096
INFO 05-09 09:34:15 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-09 09:34:15 [vllm.py:840] Asynchronous scheduling is enabled.
INFO 05-09 09:34:15 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native

generation_config.json:   0%|          | 0.00/161 [00:00<?, ?B/s]

(EngineCore pid=2202852) INFO 05-09 09:34:18 [core.py:109] Initializing a V1 LLM engine (v0.20.1) with config: model='Qwen/Qwen2.5-Math-7B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-Math-7B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=2, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, quantization_config=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, 

(EngineCore pid=2202852) Process EngineCore:
(EngineCore pid=2202852) Traceback (most recent call last):
(EngineCore pid=2202852)   File "/mnt/phongdq/miniconda3/envs/vllm/lib/python3.10/multiprocessing/process.py", line 314, in _bootstrap
(EngineCore pid=2202852)     self.run()
(EngineCore pid=2202852)   File "/mnt/phongdq/miniconda3/envs/vllm/lib/python3.10/multiprocessing/process.py", line 108, in run
(EngineCore pid=2202852)     self._target(*self._args, **self._kwargs)
(EngineCore pid=2202852)   File "/mnt/phongdq/miniconda3/envs/vllm/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 1140, in run_engine_core
(EngineCore pid=2202852)     raise e
(EngineCore pid=2202852)   File "/mnt/phongdq/miniconda3/envs/vllm/lib/python3.10/site-packages/vllm/v1/engine/core.py", line 1110, in run_engine_core
(EngineCore pid=2202852)     engine_core = EngineCoreProc(*args, engine_index=dp_rank, **kwargs)
(EngineCore pid=2202852)   File "/mnt/phongdq/miniconda3/envs/vllm/lib/python3.10/sit

RuntimeError: Engine core initialization failed. See root cause above. Failed core proc(s): {'EngineCore': 1}